# da 신뢰/비신뢰 리뷰 + SAM 보정 (Anki 스타일, 6-앙상블 기반)

새로 수집한 raw 프레임에 6-앙상블(신규 부트스트랩 5개 + 기존 배포모델, PROGRESS.md §2.20)로
da를 추론해서 오버레이로 보여주고, 사람이 **키보드 1(신뢰)/2(버리기)**로 빠르게 판단.

- **1(신뢰)**: 앙상블 예측을 그대로 채택하고 다음 프레임으로.
- **2(비신뢰)**: 그냥 버리지 않고 **SAM2 클릭 보정 모드로 즉시 전환** — 포함/제외 점을
  찍어서 그 프레임만 다시 라벨링. 다 됐으면 저장, 그래도 안 되겠으면 완전히 버림.
  SAM 연동은 최윤성님이 만든 `kuac_lane_prelabel_shared.ipynb`(SAM2+YOLOPv2 pre-labeling,
  2026-08-11 커밋)의 포인트 클릭(포함/제외) → 마스크 생성 → 수락 흐름을 참고해서 반영함
  (다만 그 노트북은 Colab+facebookresearch/sam2 레포 클론 방식이고, 여기서는 이미 로컬에
  있는 `sam2.1_b.pt` + `ultralytics.SAM` 래퍼로 동일한 멀티포인트 프롬프트를 구현 — 별도
  레포 클론/설치 없이 바로 됨).
- **주의(PROGRESS.md §2.9)**: SAM을 **자동/무보정으로** da·ll에 쓰는 건 이미 실패로
  결론남(벽/바닥까지 번짐, 완벽한 GT를 점 하나로 프롬프트해도 IoU가 오히려 떨어짐).
  여기서는 그거랑 다르게 **사람이 여러 점을 직접 찍고 눈으로 확인하며 반복 조정**하는
  거라 성격이 다름 — 그래도 검증된 적은 없는 시도이니, 실제로 잘 되는지는 써보면서
  판단할 것.
- 신뢰/보정 프레임 둘 다 `trust_review_output/trusted/`에 images+da_masks로 export
  (ll은 범위 밖 — 병합 전에 YOLOPv2+skeleton으로 별도 생성할 것)
- 진행 중 끊겨도 `trust_results.csv` + 마스크 캐시(`mask_cache/`)에 계속 저장되니
  다시 열면 이어서 진행됨
- 로컬 실행 전제(이 레포 루트에서 실행, `.venv_yolo` 커널) — 앙상블/SAM 모델이 이미
  로컬에 있어서 Colab/Drive 안 씀. 팀원과 같이 하고 싶으면 `demo.launch(share=True)`가
  만드는 공개 링크를 공유하면 됨.

## 0. 경로 설정 — INPUT_DIR을 실제 수집된 신규 프레임 폴더로 바꿀 것

In [ ]:
import os, sys, csv
from argparse import Namespace

import cv2
import numpy as np
import onnxruntime as ort
import torch

BASE = os.getcwd()  # 이 노트북을 fine-tune 레포 루트에서 실행한다고 가정

# ↓↓↓ TODO: 실제 신규 수집 프레임 폴더로 바꿀 것 (예: 지그재그 주행 raw 프레임) ↓↓↓
INPUT_DIR = os.path.join(BASE, "new_raw_frames")
OUTPUT_DIR = os.path.join(BASE, "trust_review_output")
RESULTS_CSV = os.path.join(OUTPUT_DIR, "trust_results.csv")
MASK_CACHE_DIR = os.path.join(OUTPUT_DIR, "mask_cache")  # 최종 채택된 da 마스크 즉시 저장(크래시 대비)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MASK_CACHE_DIR, exist_ok=True)

ENSEMBLE_DIR = os.path.join(BASE, "outputs", "ensemble_bootstrap_v1")
DEPLOYED_ONNX = os.path.join(BASE, "outputs", "models", "best.onnx")
SAM_CKPT = os.path.join(BASE, "sam2.1_b.pt")
N_SEEDS = 5
CONFIG = "medium"
IN_W, IN_H = 640, 384

sys.path.insert(0, os.path.join(BASE, "_TwinLiteNetPlus_ref"))
from model.model import TwinLiteNetPlus

assert os.path.isdir(INPUT_DIR), f"{INPUT_DIR} 없음 - 신규 수집 프레임 폴더로 바꿀 것"
image_files = sorted(f for f in os.listdir(INPUT_DIR) if f.lower().endswith((".png", ".jpg", ".jpeg")))
print(f"대상 이미지 {len(image_files)}장 ({INPUT_DIR})")

## 1. 6-앙상블 모델 로드 (신규 부트스트랩 5개 + 기존 배포모델 1개)

In [ ]:
torch_models = []
for seed in range(N_SEEDS):
    m = TwinLiteNetPlus(Namespace(config=CONFIG))
    state = torch.load(os.path.join(ENSEMBLE_DIR, f"seed{seed}", "best.pth"), map_location="cpu")
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    m.load_state_dict(state)
    m.eval()
    torch_models.append(m)

onnx_sess = ort.InferenceSession(DEPLOYED_ONNX, providers=["CPUExecutionProvider"])
print(f"6-앙상블 로드 완료 (신규 부트스트랩 {N_SEEDS}개 + 기존 배포모델 1개)")

## 2. da 소프트보팅 추론 + 오버레이

In [ ]:
def softmax_fg_np(l):
    m = l.max(axis=0, keepdims=True)
    e = np.exp(l - m)
    return (e / e.sum(axis=0, keepdims=True))[1]


def to_blob(img0):
    r = cv2.resize(img0, (IN_W, IN_H))
    rgb = cv2.cvtColor(r, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return np.ascontiguousarray(np.transpose(rgb, (2, 0, 1))[None, ...])


def infer_da_ensemble(img0):
    h, w = img0.shape[:2]
    blob_np = to_blob(img0)
    blob_t = torch.from_numpy(blob_np).float()
    probs = []
    for m in torch_models:
        with torch.no_grad():
            out_da, _ = m(blob_t)
        probs.append(torch.softmax(out_da, dim=1)[0, 1].numpy())
    da_out, _ = onnx_sess.run(["da", "ll"], {"images": blob_np})
    probs.append(softmax_fg_np(da_out[0]))
    avg = np.mean(probs, axis=0)
    return cv2.resize(avg, (w, h), interpolation=cv2.INTER_LINEAR) >= 0.5


def overlay_da(img_bgr, da_mask):
    vis = img_bgr.copy()
    vis[da_mask] = (0.55 * vis[da_mask].astype(np.float64) + 0.45 * np.array([0, 200, 0])).astype(np.uint8)
    return cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)

## 3. SAM2 로드 (비신뢰 프레임 클릭 보정용)

최윤성님의 `kuac_lane_prelabel_shared.ipynb`와 같은 SAM2.1 계열이지만, 여기서는
`facebookresearch/sam2` 레포를 따로 클론하지 않고 이미 로컬에 있는 `sam2.1_b.pt`를
`ultralytics.SAM` 래퍼로 로드 — 멀티 포인트(포함/제외 혼합) 프롬프트 동일하게 지원됨.

In [ ]:
from ultralytics import SAM

assert os.path.isfile(SAM_CKPT), f"{SAM_CKPT} 없음"
sam_model = SAM(SAM_CKPT)
print("SAM2(ultralytics wrapper) 로드 완료:", SAM_CKPT)


def draw_points_overlay(img_bgr, points, mask=None):
    vis = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).copy()
    if mask is not None:
        overlay = vis.copy()
        overlay[mask] = (0, 255, 0)
        vis = cv2.addWeighted(overlay, 0.4, vis, 0.6, 0)
    for x, y, lbl in points:
        color = (0, 200, 255) if lbl == 1 else (255, 0, 0)
        cv2.circle(vis, (int(x), int(y)), 6, color, -1)
    return vis


def run_sam(img_bgr, points):
    if not points:
        return None
    pts = [[p[0], p[1]] for p in points]
    lbls = [p[2] for p in points]
    res = sam_model(img_bgr, points=pts, labels=lbls, verbose=False)
    r = res[0]
    if r.masks is None or len(r.masks.data) == 0:
        return None
    best_idx = int(torch.argmax(r.boxes.conf)) if r.boxes is not None and len(r.boxes.conf) else 0
    return r.masks.data[best_idx].cpu().numpy().astype(bool)

## 4. 상태/이어하기 — 기존 `trust_results.csv`/`mask_cache` 있으면 자동으로 이어서 진행

In [ ]:
decisions = {}  # fname -> "trust" | "corrected" | "discard"
if os.path.isfile(RESULTS_CSV):
    with open(RESULTS_CSV) as f:
        for row in csv.DictReader(f):
            decisions[row["file"]] = row["decision"]
    print(f"기존 진행상황 {len(decisions)}건 불러옴")

remaining = [f for f in image_files if f not in decisions]
print(f"남은 {len(remaining)}장 / 전체 {len(image_files)}장")

state = {"idx": 0}
mode = {"m": "review"}  # "review" | "correct"
correction_state = {"points": [], "mask": None}
_da_cache = {}  # fname -> da mask(채택된 것: 앙상블 or SAM보정), export 단계에서도 씀


def _current_fname():
    return remaining[state["idx"]] if state["idx"] < len(remaining) else None


def _load_review_overlay(fname):
    if fname is None:
        return None
    img = cv2.imread(os.path.join(INPUT_DIR, fname))
    if fname not in _da_cache:
        _da_cache[fname] = infer_da_ensemble(img)
    return overlay_da(img, _da_cache[fname])


def _status():
    n_trust = sum(1 for v in decisions.values() if v == "trust")
    n_corrected = sum(1 for v in decisions.values() if v == "corrected")
    n_discard = sum(1 for v in decisions.values() if v == "discard")
    fname = _current_fname()
    if fname is None:
        return f"전부 완료! 신뢰:{n_trust} 보정:{n_corrected} 버림:{n_discard}"
    idx1 = state["idx"] + 1
    return f"{idx1} / {len(remaining)} - {fname}  |  신뢰:{n_trust} 보정:{n_corrected} 버림:{n_discard}"


def _finalize(fname, decision, mask=None):
    """최종 판정 1건을 기록: CSV에 append + (있으면) 마스크를 mask_cache에 즉시 저장."""
    decisions[fname] = decision
    is_new = not os.path.isfile(RESULTS_CSV)
    with open(RESULTS_CSV, "a", newline="") as f:
        w = csv.writer(f)
        if is_new:
            w.writerow(["file", "decision"])
        w.writerow([fname, decision])
    if mask is not None:
        _da_cache[fname] = mask
        cv2.imwrite(os.path.join(MASK_CACHE_DIR, fname), (mask.astype(np.uint8) * 255))
    state["idx"] += 1

## 5. Gradio 앱 — 리뷰(1=신뢰/2=버리기) + 비신뢰 시 SAM 클릭 보정 모드로 전환

In [ ]:
import gradio as gr

KEY_JS = """
<script>
document.addEventListener(\'keydown\', function(e) {
  if (e.key === \'1\') {
    const el = document.getElementById(\'trust_btn\');
    if (el) { (el.querySelector(\'button\') || el).click(); }
  } else if (e.key === \'2\') {
    const el = document.getElementById(\'discard_btn\');
    if (el) { (el.querySelector(\'button\') || el).click(); }
  }
});
</script>
"""


def trust():
    if mode["m"] != "review":
        return gr.update(), gr.update(), gr.update(), gr.update()
    fname = _current_fname()
    if fname is not None:
        _finalize(fname, "trust", mask=_da_cache.get(fname))
    return (gr.update(visible=True), gr.update(visible=False),
            _load_review_overlay(_current_fname()), _status())


def enter_correction():
    if mode["m"] != "review":
        return gr.update(), gr.update(), gr.update(), gr.update()
    fname = _current_fname()
    if fname is None:
        return (gr.update(visible=True), gr.update(visible=False), None, _status())
    mode["m"] = "correct"
    correction_state["points"], correction_state["mask"] = [], None
    img = cv2.imread(os.path.join(INPUT_DIR, fname))
    return (gr.update(visible=False), gr.update(visible=True),
            draw_points_overlay(img, []), f"[보정] {fname} — 클릭으로 포함(초록점)/제외(빨간점) 표시 후 \'마스크 생성\'")


def on_click_point(point_mode, evt: gr.SelectData):
    if mode["m"] != "correct":
        return gr.update()
    fname = _current_fname()
    x, y = evt.index
    label = 1 if point_mode == "포함 (da)" else 0
    correction_state["points"].append((x, y, label))
    img = cv2.imread(os.path.join(INPUT_DIR, fname))
    return draw_points_overlay(img, correction_state["points"], correction_state["mask"])


def gen_mask():
    if mode["m"] != "correct":
        return gr.update()
    fname = _current_fname()
    img = cv2.imread(os.path.join(INPUT_DIR, fname))
    mask = run_sam(img, correction_state["points"])
    correction_state["mask"] = mask
    return draw_points_overlay(img, correction_state["points"], mask)


def reset_points():
    if mode["m"] != "correct":
        return gr.update()
    correction_state["points"], correction_state["mask"] = [], None
    fname = _current_fname()
    img = cv2.imread(os.path.join(INPUT_DIR, fname))
    return draw_points_overlay(img, [])


def accept_correction():
    if mode["m"] != "correct":
        return gr.update(), gr.update(), gr.update(), gr.update()
    fname = _current_fname()
    if correction_state["mask"] is not None:
        _finalize(fname, "corrected", mask=correction_state["mask"])
    else:
        _finalize(fname, "discard")
    mode["m"] = "review"
    correction_state["points"], correction_state["mask"] = [], None
    return (gr.update(visible=True), gr.update(visible=False),
            _load_review_overlay(_current_fname()), _status())


def give_up():
    if mode["m"] != "correct":
        return gr.update(), gr.update(), gr.update(), gr.update()
    fname = _current_fname()
    _finalize(fname, "discard")
    mode["m"] = "review"
    correction_state["points"], correction_state["mask"] = [], None
    return (gr.update(visible=True), gr.update(visible=False),
            _load_review_overlay(_current_fname()), _status())


with gr.Blocks() as demo:
    gr.Markdown("### da 신뢰/비신뢰 리뷰 — 앙상블 예측이 맞으면 **1(신뢰)**, 틀리면 **2(비신뢰→SAM 보정)**")
    gr.HTML(KEY_JS)
    status = gr.Textbox(value=_status(), label="진행상황", interactive=False)

    with gr.Group(visible=True) as review_group:
        review_img = gr.Image(value=_load_review_overlay(_current_fname()), interactive=False)
        with gr.Row():
            discard_btn = gr.Button("\u2717 비신뢰 -> SAM 보정 (2)", elem_id="discard_btn")
            trust_btn = gr.Button("\u2713 신뢰 (1)", elem_id="trust_btn", variant="primary")

    with gr.Group(visible=False) as correct_group:
        gr.Markdown("클릭한 점: 초록=포함(da) / 빨강=제외(배경) — 이미지 클릭으로 점 추가")
        point_mode = gr.Radio(["포함 (da)", "제외 (배경/콘)"], value="포함 (da)", label="다음 클릭 모드")
        correct_img = gr.Image(interactive=True)
        with gr.Row():
            gen_btn = gr.Button("마스크 생성")
            reset_btn = gr.Button("점 초기화")
            giveup_btn = gr.Button("그래도 버리기")
            accept_btn = gr.Button("보정 저장하고 다음 ->", variant="primary")

    trust_btn.click(trust, outputs=[review_group, correct_group, review_img, status])
    discard_btn.click(enter_correction, outputs=[review_group, correct_group, correct_img, status])
    correct_img.select(on_click_point, inputs=[point_mode], outputs=[correct_img])
    gen_btn.click(gen_mask, outputs=[correct_img])
    reset_btn.click(reset_points, outputs=[correct_img])
    accept_btn.click(accept_correction, outputs=[review_group, correct_group, review_img, status])
    giveup_btn.click(give_up, outputs=[review_group, correct_group, review_img, status])

demo.launch(share=True, debug=False)

## 6. Export — 신뢰/보정 처리된 프레임만 images/da_masks로 저장

`bootstrap_v2`와 같은 폴더 구조(`images/`, `da_masks/`)로 나오니, 그대로 복사해서
사람검증 corpus에 병합하면 됨. **ll_masks는 여기서 안 만듦** — 병합 전에
`scripts/pseudo_label/build_pseudo_label_dataset_v2.py`류의 YOLOPv2+skeleton
로직으로 따로 생성할 것. 순수 "버림(discard)" 처리된 프레임은 export 안 됨(데이터로 안 씀).

In [ ]:
EXPORT_DIR = os.path.join(OUTPUT_DIR, "trusted")
os.makedirs(os.path.join(EXPORT_DIR, "images"), exist_ok=True)
os.makedirs(os.path.join(EXPORT_DIR, "da_masks"), exist_ok=True)

n = 0
for fname, decision in decisions.items():
    if decision not in ("trust", "corrected"):
        continue
    img = cv2.imread(os.path.join(INPUT_DIR, fname))
    da_mask = _da_cache.get(fname)
    if da_mask is None:
        mask_cache_path = os.path.join(MASK_CACHE_DIR, fname)
        if os.path.isfile(mask_cache_path):
            da_mask = cv2.imread(mask_cache_path, cv2.IMREAD_GRAYSCALE) > 127
        else:
            da_mask = infer_da_ensemble(img)  # 폴백(정상 흐름에서는 안 탐)
    cv2.imwrite(os.path.join(EXPORT_DIR, "images", fname), img)
    cv2.imwrite(os.path.join(EXPORT_DIR, "da_masks", fname), (da_mask.astype(np.uint8) * 255))
    n += 1

n_trust = sum(1 for v in decisions.values() if v == "trust")
n_corrected = sum(1 for v in decisions.values() if v == "corrected")
print(f"export 완료: {n}장 (신뢰 {n_trust} + SAM보정 {n_corrected}) -> {EXPORT_DIR}")